In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scripts.sentinel_1.pipeline import run_all_bursts
import os
from scripts.sentinel_1.pipeline import run_gamma_dop2d_pipeline, plot_comparison
from scripts.sentinel_1.doppler_comparison import plot_rvl_comparison

In [ ]:
# ── Data paths ──────────────────────────────────────────────────────────────
ROOT        = os.path.dirname(os.getcwd())          # repo root
DATA        = os.path.join(ROOT, 'ocean_current_retrieval/data')
SCENE       = f"scene{1}"

SLC_SAFE    = os.path.join(DATA, 'sentinel-1', SCENE,
    'S1A_IW_SLC.SAFE')
OCN_SAFE    = os.path.join(DATA, 'sentinel-1', SCENE,
    'S1A_IW_OCN.SAFE')
AUX_CAL     = os.path.join(DATA, 'sentinel-1',
    'S1A_AUX_CAL_V20190228T092500_G20240327T102320.SAFE')
POEORB      = os.path.join(DATA, 'sentinel-1',
    'S1A_OPER_AUX_POEORB_OPOD_20260225T070420_V20260204T225942_20260206T005942.EOF')
ERA5_WIND   = os.path.join(DATA, 'era5_data', SCENE, 'era5_wind.nc')
ERA5_WAVE   = os.path.join(DATA, 'era5_data', SCENE, 'era5_wave.nc')
GLO12       = os.path.join(DATA, 'era5_data', SCENE,
    'glo12.nc')

print('All paths exist:', all(os.path.exists(p)
    for p in [SLC_SAFE, AUX_CAL, POEORB, ERA5_WIND, ERA5_WAVE, GLO12]))

In [ ]:
results_gamma = run_gamma_dop2d_pipeline(
      dop2d_npz      = 'data/sentinel-1/gamma_iw1/20260205_iw1_vv.dop2d.npz',
      annotation_xml = 'data/sentinel-1/scene1/S1A_IW_SLC.SAFE/annotation/s1a-iw1-slc-vv-20260205t165251-20260205t165319-063086-07eaef-004.xml',
      subswath       = 'iw1',
      poeorb_path    = POEORB,
      aux_cal_path   = AUX_CAL,
      ocn_safe       = OCN_SAFE,
      era5_wind      = ERA5_WIND,
      era5_wave      = ERA5_WAVE,
      glo12          = GLO12,
)

In [ ]:
results_ocn = run_all_bursts(SLC_SAFE, 'iw1', POEORB, AUX_CAL, OCN_SAFE, ERA5_WIND, ERA5_WAVE, GLO12, 'vv', merge_first=True, deramp_method='esa_eq1', use_ocn_dc=True)

In [ ]:
results = run_all_bursts(SLC_SAFE, 'iw1', POEORB, AUX_CAL, OCN_SAFE, ERA5_WIND, ERA5_WAVE, GLO12, 'vv', merge_first=False, deramp_method='esa_eq1', block_az=64, stride_az=32, block_rg=256, stride_rg=128, use_ocn_dc=False)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 5))

d = np.load('data/sentinel-1/gamma_iw1/20260205_iw1_vv.dop2d.npz')
img1 = ax[0].imshow(d['fd_measured'], aspect='auto',
    extent=[d['range_m'][0], d['range_m'][-1],
            d['az_time_s'][-1], d['az_time_s'][0]], vmin=-20, vmax=15)
ax[0].set_xlabel('Slant range (m)')
ax[0].set_ylabel('Azimuth time (s)')
ax[0].set_title('GAMMA RS')
ax[0].invert_yaxis()
fig.colorbar(img1, ax=ax[0], label='Doppler centroid (Hz)')

img2 = ax[1].imshow(results[0]["f_dc"], aspect='auto',vmin=-5, vmax=50)
ax[1].set_xlabel('range block')
ax[1].set_ylabel('Azimuth block')
ax[1].set_title('own pipeline')
ax[1].invert_yaxis()
fig.colorbar(img2, ax=ax[1], label='Doppler centroid (Hz)')

img3 = ax[2].imshow(results[0]["f_dc_ocn"], aspect='auto', vmin=-5, vmax=50)
ax[2].set_ylabel('Azimuth block')
ax[2].set_title('OCN product')
ax[2].set_title('OCN product')
ax[2].invert_yaxis()
fig.colorbar(img3, ax=ax[2], label='Doppler centroid (Hz)')

plt.tight_layout()
plt.show()

In [ ]:
plot_comparison([results_gamma])

In [ ]:
plot_comparison(results_ocn)

In [ ]:
plot_comparison(results)

In [ ]:
b = results[4]   # or whichever burst
import numpy as np

f_res = b['f_dc'] - b['f_data_poly']
print(f"lag-1 residual:  [{np.nanmin(f_res):.1f}, {np.nanmax(f_res):.1f}] Hz  mean={np.nanmean(f_res):.1f}")
print(f"data_poly:       [{np.nanmin(b['f_data_poly']):.1f}, {np.nanmax(b['f_data_poly']):.1f}] Hz")
print(f"f_dc (fixed):    [{np.nanmin(b['f_dc']):.1f}, {np.nanmax(b['f_dc']):.1f}] Hz")
print(f"f_dc_ocn:        [{np.nanmin(b['f_dc_ocn']):.1f}, {np.nanmax(b['f_dc_ocn']):.1f}] Hz")

In [ ]:
plot_rvl_comparison(results)

In [ ]:
r = results[5]
import matplotlib.pyplot as plt
import numpy as np

our  = np.nanmean(r['f_dc'], axis=0)      # range profile
ocn  = np.nanmean(r['f_dc_ocn'], axis=0)

plt.figure()
plt.plot(our,  label='our f_dc')
plt.plot(ocn,  label='f_dc_ocn (OCN)')
plt.xlabel('range sample')
plt.ylabel('Hz')
plt.legend()
plt.title('Burst 4 — range profile of raw Doppler')
plt.show()

In [ ]:
from scripts.sentinel_1.rvl import _burst_mid_time, _eval_poly
from scripts.sentinel_1.safe_io import slant_range_time_vector, _nearest_estimate

tau = slant_range_time_vector(annot)

for i in range(len(results)):
    burst_mid = _burst_mid_time(annot, i)
    dc = _nearest_estimate(annot.dc_estimates, burst_mid)
    f_dc_annot = _eval_poly(dc.data_poly, dc.t0, tau)
    r = results[i]
    print(f"burst {i}:  annot_dc [{f_dc_annot.min():+.1f}, {f_dc_annot.max():+.1f}]  "
        f"our_f_dc [{np.nanmin(r['f_dc']):+.1f}, {np.nanmax(r['f_dc']):+.1f}]  "
        f"f_dc_ocn [{np.nanmin(r['f_dc_ocn']):+.1f}, {np.nanmax(r['f_dc_ocn']):+.1f}]")
    print(f"f_geom_poe [{r['f_geom_poe'].min():+.1f}, {r['f_geom_poe'].max():+.1f}]  "
            f"f_dca [{np.nanmin(r['f_dca']):+.1f}, {np.nanmax(r['f_dca']):+.1f}]")